# Gaia@AIP Access Example

This notebook demonstrates the minimal workflow for retrieving Gaia DR3 XP continuous mean-spectrum coefficients through Gaia@AIP. It queries a short list of Gaia DR3 `source_id` values, creates a TAP async job, uses the Simple Join Service to join those IDs to `gaiadr3.xp_continuous_mean_spectrum`, and saves the returned VOTable locally.

Credentials are read from the environment variable `GAIA_AIP_TOKEN`, an optional `.env` file, or a local `.gaia_aip_token` file.


## Imports

The notebook uses `requests` for authenticated HTTP calls, `pyvo` for the Gaia@AIP Simple Join Service, and `astropy` for reading VOTable outputs.


In [ ]:
import io
import os
import time
from pathlib import Path
from typing import Optional

import pandas as pd
import pyvo as vo
import requests
from astropy.table import Table


## Configuration

Edit `SOURCE_IDS` for a different example target list. Output files are written under `GaiaAIP/out_gaiaaip_access_example/` when the notebook is run from the repository root, or under `out_gaiaaip_access_example/` when run from inside `GaiaAIP/`.


In [ ]:
TAP_URL = "https://gaia.aip.de/tap"
SJS_URL = "https://gaia.aip.de/uws/simple-join-service"
XP_JOIN_TABLE = "gaiadr3.xp_continuous_mean_spectrum"

ROOT = Path.cwd().resolve()
if (ROOT / "GaiaAIP").exists():
    NOTEBOOK_DIR = ROOT / "GaiaAIP"
else:
    NOTEBOOK_DIR = ROOT

OUT_DIR = NOTEBOOK_DIR / "out_gaiaaip_access_example"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_IDS = [
    5853498713190525696,
]

MAX_WAIT_TAP_S = 240
MAX_WAIT_SJS_S = 300
POLL_INTERVAL_S = 1.5

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("OUT_DIR     :", OUT_DIR)


## Authentication

Set credentials before running network cells. Accepted forms are either the raw token or the full `Token <token>` string.

Recommended options:

```bash
export GAIA_AIP_TOKEN="your_token_here"
```

or create a local `.gaia_aip_token` file.


In [ ]:
PLACEHOLDER_TOKENS = {"", "YOUR_TOKEN_GOES_HERE", "<YOUR_TOKEN>", "<TOKEN>", "TOKEN"}


def load_dotenv_if_available() -> None:
    try:
        from dotenv import load_dotenv
    except Exception:
        return
    load_dotenv(NOTEBOOK_DIR / ".env")
    load_dotenv()


def normalize_gaia_aip_token(raw_token: str) -> str:
    token = (raw_token or "").strip()
    if len(token) >= 2 and token[0] in {"'", '"'} and token[-1] == token[0]:
        token = token[1:-1].strip()
    if token.upper() in PLACEHOLDER_TOKENS:
        raise RuntimeError(
            "Gaia@AIP token is missing. Set GAIA_AIP_TOKEN, define it in a local .env file, "
            "or create a .gaia_aip_token file. Do not paste tokens directly into the notebook."
        )
    if any(ord(char) > 127 for char in token):
        raise RuntimeError("GAIA_AIP_TOKEN must contain ASCII characters only. Check copied whitespace or quotes.")
    return token if token.startswith("Token ") else f"Token {token}"


def load_gaia_aip_token() -> str:
    load_dotenv_if_available()
    raw_token = os.getenv("GAIA_AIP_TOKEN", "").strip()
    token_file = Path(os.getenv("GAIA_AIP_TOKEN_FILE", str(NOTEBOOK_DIR / ".gaia_aip_token"))).expanduser()
    if not raw_token and token_file.exists():
        raw_token = token_file.read_text(encoding="utf-8").strip()
    return normalize_gaia_aip_token(raw_token)


def make_gaia_aip_session() -> requests.Session:
    session = requests.Session()
    session.headers["Authorization"] = load_gaia_aip_token()
    return session


def raise_for_aip_response(response: requests.Response, context: str) -> None:
    if response.status_code in {401, 403}:
        raise RuntimeError(
            f"Gaia@AIP rejected {context} with HTTP {response.status_code}. "
            "Check the Gaia@AIP token and restart the kernel after changing credentials."
        )
    response.raise_for_status()


## Gaia@AIP Job Helpers

Gaia@AIP uses asynchronous jobs. The helper functions below create and monitor TAP and SJS jobs, then download the final VOTable result.


In [ ]:
def wait_for_uws_job(session: requests.Session, job_url: str, *, timeout_s: int, label: str) -> None:
    start = time.time()
    while True:
        response = session.get(f"{job_url.rstrip('/')}/phase", timeout=60)
        raise_for_aip_response(response, f"{label} phase check")
        phase = response.text.strip()
        if phase in {"COMPLETED", "ERROR", "ABORTED"}:
            break
        if time.time() - start > timeout_s:
            raise TimeoutError(f"{label} did not finish within {timeout_s} seconds.")
        time.sleep(POLL_INTERVAL_S)
    if phase != "COMPLETED":
        raise RuntimeError(f"{label} ended with phase={phase}.")


def tap_create_async_job(session: requests.Session, query: str, *, runid: str) -> str:
    payload = {
        "REQUEST": "doQuery",
        "LANG": "ADQL",
        "FORMAT": "votable",
        "QUERY": query.strip().rstrip(";"),
        "RUNID": runid,
    }
    response = session.post(f"{TAP_URL}/async", data=payload, allow_redirects=False, timeout=120)
    if response.status_code in {302, 303} and "Location" in response.headers:
        job_url = response.headers["Location"]
        if job_url.startswith("/"):
            job_url = "https://gaia.aip.de" + job_url
    else:
        raise_for_aip_response(response, "TAP async job creation")
        raise RuntimeError(f"Unexpected TAP async response: HTTP {response.status_code}; body={response.text[:300]!r}")

    run_response = session.post(f"{job_url.rstrip('/')}/phase", data={"PHASE": "RUN"}, timeout=60)
    raise_for_aip_response(run_response, "TAP async job start")
    wait_for_uws_job(session, job_url, timeout_s=MAX_WAIT_TAP_S, label="Gaia@AIP TAP job")
    return job_url


def sjs_download(session: requests.Session, tap_job_url: str, join_table: str, output_dir: Path) -> Path:
    tap_job_id = tap_job_url.rstrip("/").split("/")[-1]
    service = vo.dal.DALService(SJS_URL, session=session)
    query = service.create_query(
        job_id=tap_job_id,
        column_name="source_id",
        responseformat="votable",
        join_table=join_table,
        data_structure="COMBINED",
    )
    response = query.submit(post=True)
    raise_for_aip_response(response, "SJS job creation")
    job = vo.io.uws.parse_job(io.BytesIO(response.content))
    job_url = f"{service._baseurl}/{job.jobid}"

    run_response = service._session.post(f"{job_url}/phase", data={"PHASE": "RUN"}, stream=True, timeout=60)
    raise_for_aip_response(run_response, "SJS job start")
    wait_for_uws_job(service._session, job_url, timeout_s=MAX_WAIT_SJS_S, label="Gaia@AIP SJS job")

    finished_job = vo.io.uws.parse_job(service._session.get(job_url, timeout=60).raw)
    if not finished_job.results:
        raise RuntimeError("SJS job completed but returned no result links.")

    result_url = finished_job.results[0].reference
    result_response = service._session.get(result_url, stream=True, timeout=240)
    raise_for_aip_response(result_response, "SJS result download")

    output_path = output_dir / f"sjs_{job.jobid}_result.vot"
    with output_path.open("wb") as handle:
        for chunk in result_response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                handle.write(chunk)
    return output_path


def read_votable(path: Path) -> pd.DataFrame:
    table = Table.read(path)
    return table.to_pandas()


## Run the Example Query

This cell submits the source-id list to TAP, joins it to the XP continuous mean-spectrum table, downloads the VOTable, and displays the first rows.


In [ ]:
if not SOURCE_IDS:
    raise ValueError("SOURCE_IDS is empty. Add at least one Gaia DR3 source_id before running the query.")

source_ids = [int(source_id) for source_id in SOURCE_IDS]
ids_sql = ",".join(str(source_id) for source_id in source_ids)
query = f"""
SELECT source_id
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_sql})
"""

session = make_gaia_aip_session()
tap_job_url = tap_create_async_job(session, query, runid="gaia_aip_access_example_ids")
print("TAP job URL:", tap_job_url)

votable_path = sjs_download(session, tap_job_url, XP_JOIN_TABLE, OUT_DIR)
print("Saved VOTable:", votable_path)

df_xp = read_votable(votable_path)
print("Rows:", len(df_xp))
print("Columns:", list(df_xp.columns))
df_xp.head()
